# Project Part 2 — Lossless Source Coding
## Text Source: Token-Level

In [1]:
import math, collections, heapq, os, subprocess, tempfile
import matplotlib.pyplot as plt
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

with open('Text.txt','r',encoding='utf-8') as f:
    raw = f.read()
tokens = tokenizer.encode(raw)
n = len(tokens)
avg_cpt = len(raw)/n  # avg characters per token

counts = collections.Counter(tokens)
probs  = {x:c/n for x,c in counts.items()}

def entropy(p): return -sum(v*math.log2(v) for v in p.values() if v>0)
H = entropy(probs)
print(f'Total tokens        : {n}')
print(f'Distinct tokens     : {len(probs)}')
print(f'H(X) bits/token     : {H:.4f}')
print(f'H(X) bits/character : {H/avg_cpt:.4f}')

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Token indices sequence length is longer than the specified maximum sequence length for this model (42596 > 1024). Running this sequence through the model will result in indexing errors


Total tokens        : 42596
Distinct tokens     : 680
H(X) bits/token     : 7.8656
H(X) bits/character : 1.6646


---
## Task 1 — Huffman Coding
### Q1a: Build Huffman code for token alphabet

In [2]:
def build_huffman(prob_dict):
    import heapq
    heap = [[p,i,sym,None,None] for i,(sym,p) in enumerate(prob_dict.items())]
    heapq.heapify(heap)
    nid=len(prob_dict)
    while len(heap)>1:
        l=heapq.heappop(heap);r=heapq.heappop(heap)
        heapq.heappush(heap,[l[0]+r[0],nid,None,l,r]);nid+=1
    return heap[0]

def get_codes(tree,prefix='',codes={}):
    if tree[2] is not None: codes[tree[2]]=prefix or '0'
    else: get_codes(tree[3],prefix+'0',codes);get_codes(tree[4],prefix+'1',codes)
    return codes

import heapq
tree  = build_huffman(probs)
codes = get_codes(tree,'',{})

E_l = sum(probs[s]*len(codes[s]) for s in probs)
print(f'E[l(X)] Huffman bits/token : {E_l:.4f}')
print(f'H(X) bits/token            : {H:.4f}')
print(f'E[l(X)] bits/char          : {E_l/avg_cpt:.4f}')
print(f'H(X)    bits/char          : {H/avg_cpt:.4f}')

E[l(X)] Huffman bits/token : 7.9014
H(X) bits/token            : 7.8656
E[l(X)] bits/char          : 1.6722
H(X)    bits/char          : 1.6646


### Q1b: Encode tokens and verify lossless decoding

In [3]:
sample = tokens[:2000]
bitstream = ''.join(codes[s] for s in sample)
avg_len = len(bitstream)/len(sample)

rev = {v:k for k,v in codes.items()}
decoded, buf = [], ''
for bit in bitstream:
    buf+=bit
    if buf in rev: decoded.append(rev[buf]);buf=''

print(f'Encoded 2000 tokens -> {len(bitstream)} bits')
print(f'Avg bits/token : {avg_len:.4f}')
print(f'Lossless?      : {decoded==sample}')

Encoded 2000 tokens -> 15861 bits
Avg bits/token : 7.9305
Lossless?      : True


---
## Task 2 — Block Coding
### Q2a: Block Huffman (n=2 only — alphabet grows fast!)

In [4]:
def block_huffman_avg(sym_list, block_size):
    blocks=[tuple(sym_list[i:i+block_size]) for i in range(0,len(sym_list)-block_size+1,block_size)]
    n_b=len(blocks)
    bp={b:c/n_b for b,c in collections.Counter(blocks).items()}
    tree=build_huffman(bp)
    codes_b=get_codes(tree,'',{})
    return sum(bp[b]*len(codes_b[b]) for b in bp)/block_size

print('Block Huffman (token level):')
results = []
for bs in [1, 2]:
    avg = block_huffman_avg(tokens[:5000], bs)
    results.append((bs, avg))
    print(f'  Block {bs}: {avg:.4f} bits/token = {avg/avg_cpt:.4f} bits/char')
print(f'  H(X)  : {H:.4f} bits/token = {H/avg_cpt:.4f} bits/char')
print()
print('Note: Block size 3+ is infeasible — vocab size^3 is enormous.')

Block Huffman (token level):
  Block 1: 7.9066 bits/token = 1.6733 bits/char
  Block 2: 4.7070 bits/token = 0.9961 bits/char
  H(X)  : 7.8656 bits/token = 1.6646 bits/char

Note: Block size 3+ is infeasible — vocab size^3 is enormous.


---
## Task 3 — Arithmetic Coding
### Q3a: Memoryless arithmetic coding on tokens

In [5]:
def build_cum(p):
    alph=sorted(p.keys());cum={};c=0.0
    for s in alph:cum[s]=(c,c+p[s]);c+=p[s]
    return cum

def arith_encode(seq,p):
    cum=build_cum(p);L,U,bits,pend=0.0,1.0,[],0
    for s in seq:
        if s not in cum:continue
        lo,hi=cum[s];w=U-L;U=L+w*hi;L=L+w*lo
        while True:
            if U<=0.5:   bits+=[0]+[1]*pend;pend=0;L*=2;U*=2
            elif L>=0.5: bits+=[1]+[0]*pend;pend=0;L=(L-.5)*2;U=(U-.5)*2
            elif L>=.25 and U<=.75:pend+=1;L=(L-.25)*2;U=(U-.25)*2
            else:break
    pend+=1
    if L<.25:bits+=[0]+[1]*pend
    else:bits+=[1]+[0]*pend
    return bits

sample_ac = tokens[:500]
bits_ml   = arith_encode(sample_ac, probs)
avg_ml    = len(bits_ml)/len(sample_ac)
print(f'Arithmetic (memoryless): {avg_ml:.4f} bits/token = {avg_ml/avg_cpt:.4f} bits/char')
print(f'H(X)                   : {H:.4f} bits/token = {H/avg_cpt:.4f} bits/char')

Arithmetic (memoryless): 8.0680 bits/token = 1.7074 bits/char
H(X)                   : 7.8656 bits/token = 1.6646 bits/char


### Q3b: Markov arithmetic coding (k=1)

In [ ]:
trans=collections.defaultdict(collections.Counter)
for i in range(len(tokens)-1):trans[tokens[i]][tokens[i+1]]+=1
trans_p={ctx:{s:c/sum(cn.values()) for s,c in cn.items()} for ctx,cn in trans.items()}

def arith_encode_markov(seq,trans_p,init_p):
    L,U,bits,pend,prev=0.0,1.0,[],0,None
    for s in seq:
        pm=trans_p.get(prev,init_p) if prev is not None else init_p
        cum=build_cum(pm)
        if s not in cum:prev=s;continue
        lo,hi=cum[s];w=U-L;U=L+w*hi;L=L+w*lo
        while True:
            if U<=0.5:   bits+=[0]+[1]*pend;pend=0;L*=2;U*=2
            elif L>=0.5: bits+=[1]+[0]*pend;pend=0;L=(L-.5)*2;U=(U-.5)*2
            elif L>=.25 and U<=.75:pend+=1;L=(L-.25)*2;U=(U-.25)*2
            else:break
        prev=s
    pend+=1
    if L<.25:bits+=[0]+[1]*pend
    else:bits+=[1]+[0]*pend
    return bits

bits_mk = arith_encode_markov(sample_ac, trans_p, probs)
avg_mk  = len(bits_mk)/len(sample_ac)
print(f'Arithmetic Markov k=1 : {avg_mk:.4f} bits/token = {avg_mk/avg_cpt:.4f} bits/char')
print(f'Memoryless            : {avg_ml:.4f} bits/token')

---
## Task 4 — Summary
### Q4a: Full comparison table

In [ ]:
print('=== Compression — Token Level (bits/char for fair comparison) ===')
print(f'{"Method":<35} {"bits/token":>12} {"bits/char":>10}')
print('-'*60)
print(f'{"H(X)":<35} {H:>12.4f} {H/avg_cpt:>10.4f}')
print(f'{"Huffman (token)":<35} {E_l:>12.4f} {E_l/avg_cpt:>10.4f}')
for bs, avg in results:
    print(f'{f"Block Huffman n={bs}":<35} {avg:>12.4f} {avg/avg_cpt:>10.4f}')
print(f'{"Arithmetic memoryless":<35} {avg_ml:>12.4f} {avg_ml/avg_cpt:>10.4f}')
print(f'{"Arithmetic Markov k=1":<35} {avg_mk:>12.4f} {avg_mk/avg_cpt:>10.4f}')